# Week 36, Day 4: Machine learning for decoding

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w036/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The Wiener filter

Standardise the lagged counts with the training mean and standard deviation (given), then fit ridge weights (penalty 1) from Zs plus a constant column to V on the training bins, and compute the test R² for both velocity components.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2020)
dt, T, N = 0.05, 12000, 40
X = np.zeros((T, 4))
for t in range(1, T):
    p, v = X[t - 1, :2], X[t - 1, 2:]
    v = v + dt * (-2.0 * v - 1.5 * p) + np.sqrt(dt) * 12 * r.normal(size=2)
    X[t, :2] = p + dt * v; X[t, 2:] = v
V = X[:, 2:]
pd = r.uniform(0, 2 * np.pi, N); kap = r.uniform(1.5, 3, N); base = r.uniform(2, 10, N)
sp, th = np.hypot(V[:, 0], V[:, 1]), np.arctan2(V[:, 1], V[:, 0])
axial = np.arange(N) < 30
tun = np.where(axial, np.cos(2 * (th[:, None] - pd)), 0.4 * np.cos(th[:, None] - pd))
Z = r.poisson(base * np.exp(kap * tun * np.tanh(sp[:, None] / 6)) * dt).astype(float)
Lg = 3
Zl = np.concatenate([np.roll(Z, k, 0) for k in range(Lg)], 1); Zl[:Lg] = 0
tr, te = np.arange(T) < 8000, np.arange(T) >= 9000
mu, sd = Zl[tr].mean(0), Zl[tr].std(0) + 1e-9
Zs = (Zl - mu) / sd
r2 = lambda a, b: 1 - ((a - b) ** 2).sum(0) / ((a - a.mean(0)) ** 2).sum(0)

def wiener(ntr):
    A = np.c_[Zs[:ntr], np.ones(ntr)]
    B = ...
    return np.c_[Zs, np.ones(T)] @ B

r2_w = r2(V[te], wiener(8000)[te])
print("Wiener filter R squared (vx, vy):", np.round(r2_w, 3))

In [ ]:
import numpy as np
def test_wiener():
    "the Wiener filter decodes velocity with mean R squared between 0.3 and 0.6"
    assert 0.3 < r2_w.mean() < 0.6

test_wiener()
print("All checks passed")

### Exercise: Forward, backward, Adam

Complete the backward pass inside mlp(): given the batch inputs x, hidden activations h and output error d, compute dh (the error at the hidden layer) and the four gradients gW1, gb1, gW2, gb2 (add wd * W1 and wd * W2 to the weight gradients).

In [ ]:
def mlp(ntr, H=64, steps=3000, lr=3e-3, seed=0, wd=1e-3):
    g = np.random.default_rng(seed); D = Zs.shape[1]; nv = ntr // 5; nfit = ntr - nv
    W1 = g.normal(0, 1 / np.sqrt(D), (D, H)); b1 = np.zeros(H)
    W2 = g.normal(0, 1 / np.sqrt(H), (H, 2)); b2 = np.zeros(2)
    ps = [W1, b1, W2, b2]; m = [np.zeros_like(p) for p in ps]; v = [np.zeros_like(p) for p in ps]
    vs = V.std(0); best = (np.inf, None)
    predict = lambda x: (np.maximum(0, x @ W1 + b1) @ W2 + b2) * vs
    for k in range(1, steps + 1):
        j = g.integers(0, nfit, 128); x = Zs[j]; y = V[j] / vs
        h = np.maximum(0, x @ W1 + b1)
        d = (h @ W2 + b2 - y) / len(j)
        ...
        for p, gr, mm, vv in zip(ps, [gW1, gb1, gW2, gb2], m, v):
            mm *= 0.9; mm += 0.1 * gr; vv *= 0.999; vv += 0.001 * gr ** 2
            p -= lr * (mm / (1 - 0.9 ** k)) / (np.sqrt(vv / (1 - 0.999 ** k)) + 1e-8)
        if k % 100 == 0:
            e = np.mean((predict(Zs[nfit:ntr]) - V[nfit:ntr]) ** 2)
            if e < best[0]:
                best = (e, [p.copy() for p in ps])
    W1, b1, W2, b2 = best[1]
    return (np.maximum(0, Zs @ W1 + b1) @ W2 + b2) * vs

r2_m = r2(V[te], mlp(8000)[te])
print("network R squared (vx, vy):", np.round(r2_m, 3), "  Wiener:", np.round(r2_w, 3))

In [ ]:
import numpy as np
def test_beats_linear():
    "with 8,000 training bins, the network beats the Wiener filter by more than 0.1 in mean R squared"
    assert r2_m.mean() > r2_w.mean() + 0.1

test_beats_linear()
print("All checks passed")

### Exercise: A learning curve

For training sizes of 1,000, 2,000, 4,000 and 8,000 bins, compute the mean test R² of both decoders and store them in curve[n] = {'Wiener': ..., 'network': ...}. (This takes a few seconds per size.)

In [ ]:
curve = {}
for n in (1000, 2000, 4000, 8000):
    curve[n] = ...
    print(n, {k: round(float(x), 3) for k, x in curve[n].items()})
plt.plot(list(curve), [c["Wiener"] for c in curve.values()], "o-", label="Wiener")
plt.plot(list(curve), [c["network"] for c in curve.values()], "o-", label="network")
plt.xscale("log"); plt.xlabel("training bins"); plt.ylabel("test R squared"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_growth():
    "the network's advantage over the Wiener filter is larger with 8,000 training bins than with 1,000"
    adv = {n: curve[n]["network"] - curve[n]["Wiener"] for n in curve}
    assert adv[8000] > adv[1000]

def test_wiener_flat():
    "the Wiener filter gains less than 0.12 from 1,000 to 8,000 bins"
    assert curve[8000]["Wiener"] - curve[1000]["Wiener"] < 0.12

test_growth()
test_wiener_flat()
print("All checks passed")